# 01 · Exploratory Data Analysis — MMRDR colour fundus photographs (CFP)

**Purpose.** Characterise the CFP subset of MMRDR before any modelling, so that every preprocessing,
splitting and evaluation decision is justified by evidence from the data.

| § | Question | Informs |
|---|---|---|
| 1 | Are the labels balanced, and is the official split representative? | loss weighting, metrics, confidence intervals |
| 2 | Is laterality balanced across grades/splits? | the only available subgroup variable |
| 3 | How do lesion labels relate to grade? Are labels internally consistent? | multi-task design, label-noise discussion |
| 4 | What are the image properties (resolution, brightness, contrast, sharpness, field of view)? | resizing, cropping, normalisation |
| 5 | Do image properties differ by grade? (shortcut-learning risk) | confounding / limitations |
| 6 | Are there exact or near-duplicate images, especially across train/test? | leakage-safe splitting |
| 7 | What do typical and low-quality images look like? | augmentation, quality-robustness experiment |
| 8 | Which preprocessing pipeline? | final input pipeline |

All figures are written to `FIG_DIR` and all derived tables to `OUT_DIR` for reuse by the split script and the report.

In [ ]:
# ---- Configuration -----------------------------------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = "/content/drive/MyDrive/mmrdr"          # permanent copy (Drive)
ROOT       = "/content/data/MMRDR-CFP"               # fast local working copy
OUT_DIR    = f"{DRIVE_ROOT}/eda"                     # derived tables
FIG_DIR    = f"{DRIVE_ROOT}/eda/figures"             # report-ready figures
SEED = 42

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists(f"{ROOT}/FP.csv"):                     # copy once per session
        os.makedirs("/content/data", exist_ok=True)
        os.system(f'cp -r "{DRIVE_ROOT}/MMRDR-CFP" /content/data/')

os.makedirs(FIG_DIR, exist_ok=True)
print("images:", len(os.listdir(f"{ROOT}/img")))

In [ ]:
import ast, hashlib, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from PIL import Image
from scipy import stats
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

GRADES = ["0 No DR", "1 Mild", "2 Moderate", "3 Severe", "4 PDR"]
LES    = ["MA", "HE", "IH", "VB_IRMA", "NV", "VH", "RD"]
LES_FULL = ["Microaneurysm", "Hard exudate", "Intraretinal haemorrhage", "VB / IRMA",
            "Neovascularisation", "Vitreous haemorrhage", "Retinal detachment"]
def save(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png"); plt.show()

df = pd.read_csv(f"{ROOT}/FP.csv")
df["split"] = df["image"].str.extract(r"img/(tr|ts)")[0].map({"tr": "train", "ts": "test"})
L = np.array(df["lesion"].apply(ast.literal_eval).tolist(), dtype=int)
for i, k in enumerate(LES): df[k] = L[:, i]
df["n_lesions"] = L.sum(1)
assert df["split"].notna().all() and df["grade"].between(0, 4).all()
print(df.shape); df.head()

## 1 · Label distribution and the official split

In [ ]:
ct = pd.crosstab(df.grade, df.split)[["train", "test"]]
pct = (ct / ct.sum() * 100).round(1)
table = pd.concat({"n": ct, "%": pct}, axis=1); table.index = GRADES
display(table)

chi2, p, *_ = stats.chi2_contingency(ct)
print(f"Grade distribution train vs test: chi2={chi2:.2f}, p={p:.3f}  (p>0.05 ⇒ split is stratified)")
print("Imbalance ratio (largest/smallest class):", round(ct.sum(1).max() / ct.sum(1).min(), 1))
print("Referable DR (grade ≥2) prevalence: {:.1%}".format((df.grade >= 2).mean()))
print("Sight-threatening (grade ≥3) prevalence: {:.1%}".format((df.grade >= 3).mean()))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
x = np.arange(5); w = 0.38
for j, (s, c) in enumerate([("train", "#4C72B0"), ("test", "#DD8452")]):
    b = ax.bar(x + (j - .5) * w, pct[s], w, label=f"{s} (n={ct[s].sum()})", color=c)
    ax.bar_label(b, labels=ct[s], fontsize=7, padding=1)
ax.set_xticks(x, GRADES); ax.set_ylabel("% of split"); ax.legend(frameon=False)
ax.set_title("DR grade distribution in the official train/test split")
save(fig, "01_grade_distribution")

**Notes to carry forward.** Severe imbalance (majority class ≈ 59 %) ⇒ accuracy alone is misleading; report macro-F1, QWK and per-class metrics,
and use class-balanced loss. Only ~100 Severe and ~134 PDR test images ⇒ per-class estimates are noisy; report bootstrap 95 % CIs.

## 2 · Laterality (the only patient-level variable available)

In [ ]:
lat = pd.crosstab(df.grade, df.lr.map({0: "left", 1: "right"})); lat.index = GRADES
display(lat.assign(**{"% right": (lat["right"] / lat.sum(1) * 100).round(1)}))
chi2, p, *_ = stats.chi2_contingency(lat)
print(f"Laterality vs grade: chi2={chi2:.2f}, p={p:.3f}")
print("No patient ID, age, sex, ethnicity, camera or site ⇒ subgroup/bias auditing is limited to laterality.")

## 3 · Lesion labels: prevalence, co-occurrence and consistency with the grading criteria

In [ ]:
prev = df.groupby("grade")[LES].mean() * 100
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.15, 1]})
im = axes[0].imshow(prev.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
for (i, j), v in np.ndenumerate(prev.values):
    axes[0].text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 60 else "black")
axes[0].set_xticks(range(7), LES, rotation=30); axes[0].set_yticks(range(5), GRADES)
axes[0].set_title("Lesion prevalence by grade (%)"); fig.colorbar(im, ax=axes[0], fraction=.04)

corr = np.corrcoef(L.T)                                  # phi coefficient for binary labels
im2 = axes[1].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
for (i, j), v in np.ndenumerate(corr):
    axes[1].text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
axes[1].set_xticks(range(7), LES, rotation=30); axes[1].set_yticks(range(7), LES)
axes[1].set_title("Lesion co-occurrence (phi correlation)"); fig.colorbar(im2, ax=axes[1], fraction=.04)
save(fig, "02_lesions_by_grade")
display(df[LES].sum().rename(dict(zip(LES, LES_FULL))).to_frame("images with lesion"))

In [ ]:
# Consistency with the paper's stated criteria (ICDR-style):
#   0 = no lesions; 1 = MA only; 2 = MA/IH/HE beyond mild; 3 = 4-2-1 rule (VB/IRMA or severe IH);
#   4 = NV, VH or RD present.
pdr = df[["NV", "VH", "RD"]].max(1) == 1
rules = {
    "R1  grade 0 with any lesion":              (df.grade == 0) & (df.n_lesions > 0),
    "R2  grade ≥1 with no lesion":              (df.grade >= 1) & (df.n_lesions == 0),
    "R3  grade 1 with lesion other than MA":    (df.grade == 1) & (L[:, 1:].sum(1) > 0),
    "R4  grade 1 without MA":                   (df.grade == 1) & (df.MA == 0),
    "R5  grade <4 with NV/VH/RD":               (df.grade < 4) & pdr,
    "R6  grade 4 without NV/VH/RD":             (df.grade == 4) & ~pdr,
    "R7  grade <3 with VB/IRMA":                (df.grade < 3) & (df.VB_IRMA == 1),
}
denom = {"R1": df.grade == 0, "R2": df.grade >= 1, "R3": df.grade == 1, "R4": df.grade == 1,
         "R5": df.grade < 4, "R6": df.grade == 4, "R7": df.grade < 3}
audit = pd.DataFrame({"n": {k: int(v.sum()) for k, v in rules.items()},
                      "% of relevant grades": {k: round(100 * v.sum() / max(1, denom[k[:2]].sum()), 1) for k, v in rules.items()}})
display(audit)

flag = np.zeros(len(df), bool)
for v in rules.values(): flag |= v.values
df["label_flag"] = flag
print("Grade-1 images that contain hard exudate:", int(((df.grade == 1) & (df.HE == 1)).sum()),
      f"({((df.grade == 1) & (df.HE == 1)).sum() / (df.grade == 1).sum():.1%} of Mild)")
print("Combinations of lesions seen in Mild:"); display(df[df.grade == 1][LES].value_counts().head(5))

**Interpretation.** If R1–R7 are mostly zero, grades follow the lesion-based criteria closely ⇒ lesion labels are effectively the *reasons* for a grade,
supporting a multi-task (grade + lesion) model and lesion-level explanations. Any large non-zero rule (e.g. hard exudates in Mild) indicates a
**grading-convention ambiguity** (ICDR "MA only" vs. schemes that admit exudates at the mild stage, e.g. UK NSC R1) and predicts confusion between
grades 1 and 2. Inter-grader agreement in the source paper was κ ≥ 0.75, so some adjacent-grade noise is expected irrespective of the model.

## 4 · Image properties

In [ ]:
# Decode at 1/4 resolution for speed; resolution is read from the header at full size.
def props(rel):
    p = f"{ROOT}/{rel}"
    W, H = Image.open(p).size
    img = cv2.imread(p, cv2.IMREAD_REDUCED_COLOR_4)            # BGR, 1/4 scale
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    fov = g > 15                                               # retina vs black background
    gi = g[fov] if fov.any() else g.ravel()
    lap = cv2.Laplacian(g, cv2.CV_64F)[cv2.erode(fov.astype(np.uint8), np.ones((9, 9))) > 0]
    b, gch, r = [img[..., c][fov].mean() if fov.any() else 0 for c in range(3)]
    return dict(width=W, height=H, filesize_kb=os.path.getsize(p) / 1024,
                fov_fraction=fov.mean(), brightness=gi.mean(), contrast=gi.std(),
                sharpness=lap.var() if lap.size else 0.0, mean_R=r, mean_G=gch, mean_B=b,
                dark_frac=(gi < 40).mean(), sat_frac=(gi > 245).mean())

cache = f"{OUT_DIR}/cfp_image_properties.csv"
if os.path.exists(cache):
    P = pd.read_csv(cache)
else:
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(8) as ex: P = pd.DataFrame(list(ex.map(props, df.image)))
    P.insert(0, "image", df.image.values); P.to_csv(cache, index=False)
df = df.merge(P, on="image")
df["aspect"] = df.width / df.height
PROPS = ["width", "aspect", "filesize_kb", "fov_fraction", "brightness", "contrast", "sharpness"]
display(df[PROPS].describe().T.round(2))
print("Distinct resolutions:", df.groupby(["width", "height"]).ngroups,
      "| non-square (|aspect-1|>0.02):", int((abs(df.aspect - 1) > .02).sum()))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5.5)); axes = axes.ravel()
for ax, c in zip(axes, ["width", "filesize_kb", "fov_fraction", "brightness", "contrast", "sharpness", "mean_R", "mean_G"]):
    data = df[c].clip(upper=df[c].quantile(.995))
    ax.hist(data, bins=60, color="#4C72B0", alpha=.85); ax.set_title(c)
fig.suptitle("Distribution of image properties (all CFP images)"); fig.tight_layout()
save(fig, "03_image_properties")

## 5 · Do image properties differ by grade? (shortcut-learning check)
If, say, PDR images were systematically larger, darker or from a different camera, a model could learn the
*acquisition artefact* rather than the pathology. Spearman ρ between each property and grade quantifies this;
|ρ| ≥ 0.2 is treated as a material association worth reporting.

In [ ]:
rows = []
for c in PROPS + ["mean_R", "mean_G", "mean_B", "dark_frac"]:
    rho, p = stats.spearmanr(df[c], df.grade)
    ks, pks = stats.ks_2samp(df.loc[df.split == "train", c], df.loc[df.split == "test", c])
    rows.append(dict(property=c, spearman_rho_vs_grade=round(rho, 3), p=f"{p:.1e}",
                     KS_train_vs_test=round(ks, 3), KS_p=round(pks, 3)))
assoc = pd.DataFrame(rows).set_index("property"); display(assoc)

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, c in zip(axes, ["width", "brightness", "contrast", "sharpness"]):
    ax.boxplot([df.loc[df.grade == g, c].clip(upper=df[c].quantile(.99)) for g in range(5)],
               showfliers=False)
    ax.set_xticks(range(1, 6), [g.split()[0] for g in GRADES])
    ax.set_title(f"{c}  (ρ={assoc.loc[c, 'spearman_rho_vs_grade']})"); ax.set_xlabel("grade")
fig.tight_layout(); save(fig, "04_properties_by_grade")

**Interpretation.** Large |ρ| ⇒ potential shortcut; mitigate with resolution-independent preprocessing (crop to field of view, resize to a
fixed size) and photometric augmentation, and note it as a limitation. Small KS statistics train-vs-test ⇒ the official split is not
image-property shifted, so the test set does **not** measure robustness to new cameras; this motivates the synthetic image-quality experiment.

## 6 · Exact and near-duplicate images (leakage audit)

In [ ]:
df["md5"] = [hashlib.md5(open(f"{ROOT}/{p}", "rb").read()).hexdigest() for p in df.image]
ex = df[df.duplicated("md5", keep=False)]; gx = ex.groupby("md5")
print("exact (byte-identical) duplicates:", len(ex), "images in", gx.ngroups, "groups")
print("exact groups spanning train AND test:", int((gx.split.nunique() > 1).sum()),
      "| with conflicting grades:", int((gx.grade.nunique() > 1).sum()))

**Why not perceptual hashing?** A first attempt with 64-bit pHash (Zauner 2010) flagged ~87 000 pairs, including grade conflicts at
Hamming distance 0 and visually different eyes at distance 2. pHash encodes coarse layout, and every fundus photograph shares the same
layout (circular field, optic disc, dark macula), so it cannot discriminate between eyes. Instead we compare the **retinal vessel pattern**,
which is eye-specific (the basis of retinal biometrics), using cosine similarity of high-pass-filtered green-channel thumbnails.
The method is validated by checking that all known byte-identical pairs score ≈ 1.0.

In [ ]:
# ---- 6b · Near-duplicate detection on retinal vessel structure (replaces pHash) ----
# Rationale: pHash (64-bit) encodes coarse layout; all fundus images share the same layout
# (circular FOV, optic disc, dark macula), so unrelated eyes collide (see table above: 13 grade
# conflicts even at Hamming 0). The retinal vessel pattern is eye-specific (the basis of retinal
# biometrics), so we compare high-pass-filtered green-channel thumbnails with cosine similarity.
def vessel_vec(rel, size=96):
    im = cv2.imread(f"{ROOT}/{rel}", cv2.IMREAD_REDUCED_COLOR_4)
    h, w = im.shape[:2]; m = max(h, w)
    sq = np.zeros((m, m), np.float32)
    sq[(m - h) // 2:(m - h) // 2 + h, (m - w) // 2:(m - w) // 2 + w] = im[..., 1]   # green channel: best vessel contrast
    g = cv2.resize(sq, (256, 256), interpolation=cv2.INTER_AREA)
    fov = cv2.erode((g > 15).astype(np.uint8), np.ones((15, 15), np.uint8)).astype(bool)
    hp = g - cv2.GaussianBlur(g, (0, 0), 6)                                      # remove illumination, keep vessels
    hp[~fov] = 0
    v = cv2.resize(hp, (size, size), interpolation=cv2.INTER_AREA).ravel()
    v -= v.mean(); return (v / (np.linalg.norm(v) + 1e-8)).astype(np.float32)

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(8) as ex: X = np.stack(list(ex.map(vessel_vec, df.image)))

try:
    import torch; dev = "cuda" if torch.cuda.is_available() else "cpu"
    Xt = torch.from_numpy(X).to(dev)
    def sims(a, b): return (Xt[a:b] @ Xt.T).cpu().numpy()
except ImportError:
    def sims(a, b): return X[a:b] @ X.T

LOW = 0.80                                    # record every pair above this for inspection
rows, best = [], np.zeros(len(X), np.float32)
for a in range(0, len(X), 1024):
    Smat = sims(a, a + 1024)
    for r in range(Smat.shape[0]): Smat[r, a + r] = -1          # ignore self
    best[a:a + Smat.shape[0]] = Smat.max(1)
    ii, jj = np.nonzero(Smat > LOW)
    for i, j in zip(ii + a, jj):
        if i < j: rows.append((i, j, float(Smat[i - a, j])))
vp = pd.DataFrame(rows, columns=["i", "j", "sim"])
vp["exact"] = df.md5.values[vp.i] == df.md5.values[vp.j]
vp["cross_split"] = df.split.values[vp.i] != df.split.values[vp.j]
vp["grade_conflict"] = df.grade.values[vp.i] != df.grade.values[vp.j]

# Sanity check: every known exact (md5) duplicate pair must score ≈ 1.0
md5_pairs = df[df.duplicated("md5", keep=False)].groupby("md5").size()
print("exact md5 duplicate pairs found by vessel similarity:", int(vp.exact.sum()),
      "| min similarity of exact pairs:", round(vp.loc[vp.exact, "sim"].min(), 4) if vp.exact.any() else None)

bins = [0.80, 0.85, 0.90, 0.95, 0.98, 0.995, 1.01]
vp["band"] = pd.cut(vp.sim, bins)
display(vp.groupby("band")[["exact", "cross_split", "grade_conflict"]].agg(["size", "sum"]))

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.hist(best, bins=100, color="#4C72B0"); ax.set_yscale("log")
ax.set_xlabel("highest vessel-pattern similarity to any other image"); ax.set_ylabel("images (log)")
ax.set_title("Nearest-neighbour similarity: a gap separates copies from distinct eyes")
save(fig, "05a_vessel_similarity_hist")

# Visual check: two non-exact pairs per band, so the threshold is chosen by eye, not assumed
show = vp[~vp.exact].sort_values("sim", ascending=False).groupby("band", observed=True).head(2)
show = show.sort_values("sim", ascending=False).head(10)
if len(show):
    fig, axes = plt.subplots(2, len(show), figsize=(2.6 * len(show), 5.6), squeeze=False)
    for k, (_, r) in enumerate(show.iterrows()):
        for row, idx in enumerate([r.i, r.j]):
            axes[row, k].imshow(cv2.cvtColor(cv2.imread(f"{ROOT}/{df.image[idx]}", cv2.IMREAD_REDUCED_COLOR_4), cv2.COLOR_BGR2RGB))
            axes[row, k].set_title(f"{df.split[idx]} · g{df.grade[idx]}" + (f"\nsim {r.sim:.3f}" if row == 0 else ""), fontsize=8)
            axes[row, k].axis("off")
    fig.suptitle("Highest-similarity non-identical pairs (top/bottom = pair)"); save(fig, "05b_vessel_near_duplicates")

In [ ]:
# ---- 6c · Final duplicate groups (exact md5 + vessel similarity ≥ TAU) ----
TAU = 0.95      # set from the histogram gap and the visual check above (empirical, documented)
parent = list(range(len(df)))
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
for _, grp in df.groupby("md5").indices.items():
    for k in grp[1:]: parent[find(k)] = find(grp[0])
for i, j in vp.loc[vp.sim >= TAU, ["i", "j"]].values: parent[find(int(i))] = find(int(j))
df["dup_group"] = [find(i) for i in range(len(df))]
g = df.groupby("dup_group"); size = g.image.transform("size")
cross = g.split.transform("nunique") > 1
print("images in a duplicate group:", int((size > 1).sum()), "| groups:", int((g.size() > 1).sum()))
print("groups spanning train AND test:", int((g.split.nunique() > 1).sum()))
print("groups with conflicting grades:", int((g.grade.nunique() > 1).sum()))
print("test images with a copy in train:", int(((df.split == "test") & cross).sum()))
print("train images to drop (copy of a test image):", int(((df.split == "train") & cross).sum()))

**Decision.** Choose `TAU` from the gap in the similarity histogram and the visual check (true copies vs. different eyes); document it as an empirical, visually validated threshold.
Keep the **official test set intact** (comparability with published baselines), **remove training images whose duplicate group contains a
test image**, keep one image per duplicate group within train, and keep each group on one side of the train/validation split.
Report test metrics on both the full test set and the de-duplicated test subset.

## 7 · Visual inspection: typical images per grade and lowest-quality images

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for g_ in range(5):
    idx = rng.choice(df.index[df.grade == g_], 5, replace=False)
    for k, i in enumerate(idx):
        ax = axes[g_, k]
        ax.imshow(cv2.cvtColor(cv2.imread(f"{ROOT}/{df.image[i]}", cv2.IMREAD_REDUCED_COLOR_4), cv2.COLOR_BGR2RGB))
        les = ",".join(l for l in LES if df.at[i, l]) or "—"; ax.set_title(les, fontsize=7); ax.axis("off")
    axes[g_, 0].text(-0.08, .5, GRADES[g_], transform=axes[g_, 0].transAxes, rotation=90, va="center", ha="right", fontsize=10)
fig.suptitle("Random examples per grade (title = annotated lesions)"); fig.tight_layout(); save(fig, "06_examples_per_grade")

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row, (col, asc, lab) in enumerate([("sharpness", True, "least sharp"), ("brightness", True, "darkest")]):
    for k, i in enumerate(df.sort_values(col, ascending=asc).index[:6]):
        axes[row, k].imshow(cv2.cvtColor(cv2.imread(f"{ROOT}/{df.image[i]}", cv2.IMREAD_REDUCED_COLOR_4), cv2.COLOR_BGR2RGB))
        axes[row, k].set_title(f"{lab} · g{df.grade[i]}", fontsize=8); axes[row, k].axis("off")
fig.suptitle("Lowest-quality images by sharpness and brightness"); fig.tight_layout(); save(fig, "07_low_quality_examples")

## 8 · Preprocessing pipeline — derived from §1–7

| Step | Decision | Evidence from this notebook | Literature |
|---|---|---|---|
| P1 | **Pad to square with black borders; no field-of-view crop; never stretch** | §4: median FOV fraction 0.785 ≈ π/4, i.e. images are already cropped to a circle inscribed in a square; 1,290 images (11.6 %) are non-square (aspect up to 1.33, clipped circles, §7) — stretching would distort lesion geometry | Graham 2015 |
| P1b | **Set everything outside the retinal circle to black** (geometric mask) | §8: some cameras leave a grey background — a device signature the model could exploit; a brightness-threshold mask removed dark retina, so the mask uses the inscribed-circle geometry established in §4 | Geirhos 2020 |
| P2 | **Resize to 512 × 512** (area interpolation) | §4: median width 1,659 px → ×3.2 downscale; microaneurysms (≈10–20 px native) stay ≈3–6 px at 512 but fall to ≈1–3 px at 224. Matches the dataset paper's baselines | Sahlsten 2019; Tang 2026 |
| P3 | **Illumination normalisation chosen by validation ablation** (none / Graham / CLAHE) | §4/§7: brightness 26–210, colour casts (green/yellow tints), saturated red channel in a sub-cluster; 1,857 distinct resolutions ⇒ many cameras | Graham 2015; Zuiderveld 1994 |
| P4 | **Photometric augmentation** (brightness, contrast, gamma, hue/saturation) | Same evidence as P3; simulates camera variation expected across Boots stores | Gulshan 2016 |
| P5 | **Mild blur / JPEG augmentation applied to all grades** | §5: sharpness ρ = −0.23 and file size ρ = −0.21 with grade (file size tracks sharpness under JPEG). Risk: model learns "blurry ⇒ diseased" ⇒ over-grading of poorly focused healthy eyes in deployment. Augmentation decorrelates blur from grade | Geirhos 2020 |
| P6 | **Geometric augmentation**: flips, rotation ±180°, small scale jitter; no large crops | Grading has no canonical orientation; §2 laterality–grade association is weak (p = 0.042, max 5 pp), so flipping is acceptable; large crops could remove the only lesion defining a grade | Graham 2015 |
| P7 | **Do not exclude low-quality images** | §7: the darkest / least-sharp images are disproportionately PDR (vitreous haemorrhage causes haze) — exclusion would remove the most severe cases. Quality is studied instead in the robustness experiment | Hendrycks & Dietterich 2019 |
| P8 | **Robustness experiment specifically tests the §5 shortcut**: blur grade-0 test images — does the predicted grade rise? | §5 + KS train vs test ≤ 0.033: the official test set has no quality shift, so robustness must be tested synthetically | Hendrycks & Dietterich 2019; Beede 2020 |

In [ ]:
# ---- 8 · Final preprocessing + candidate normalisations + augmentation preview ----
def pad_square(img):
    h, w = img.shape[:2]; s = max(h, w)
    out = np.zeros((s, s, 3), img.dtype)
    out[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = img
    return out
def preprocess(img, size=512):                       # P1 + P1b + P2
    img = img.copy(); img[~fov_mask(img)] = 0        # uniform black outside the retina
    return cv2.resize(pad_square(img), (size, size), interpolation=cv2.INTER_AREA)
def fov_mask(img, shrink=0.99):
    # Geometric retina mask: images are a circle inscribed in the frame (median FOV fraction
    # 0.785 ≈ π/4, §4), clipped top/bottom when non-square ⇒ circle centred in the image with
    # diameter = longer side. Brightness-independent: handles dark retinas and grey backgrounds.
    h, w = img.shape[:2]
    yy, xx = np.ogrid[:h, :w]
    r = shrink * max(h, w) / 2
    return (xx - (w - 1) / 2) ** 2 + (yy - (h - 1) / 2) ** 2 <= r ** 2

def graham(img, sigma=10):                           # P3 candidate
    out = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)
    out[~(fov_mask(img) & (img.max(2) > 0))] = 0; return out   # also exclude padded strips
def clahe_lab(img):                                  # P3 candidate
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

rng = np.random.default_rng(SEED)
nonsq = df.index[abs(df.aspect - 1) > .1]
if len(nonsq) == 0: nonsq = df.index
picks = [rng.choice(nonsq),                                        # a non-square image (P1)
         df.sort_values("brightness").index[len(df) // 50],        # a dark image (P3)
         rng.choice(df.index[df.grade == 4])]
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for r, i in enumerate(picks):
    raw = cv2.imread(f"{ROOT}/{df.image[i]}"); base = preprocess(raw)
    for c, (t, im) in enumerate([(f"raw {raw.shape[1]}×{raw.shape[0]} · g{df.grade[i]}", raw),
                                 ("pad + 512 (P1–P2)", base), ("+ Graham (P3)", graham(base)), ("+ CLAHE (P3)", clahe_lab(base))]):
        axes[r, c].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")
fig.tight_layout(); save(fig, "08_preprocessing_options")

# Augmentation preview (P4–P6) — the same transforms are implemented in the training code
def augment(img, r):
    mask = fov_mask(img)
    out = img.astype(np.float32)
    out = out * r.uniform(.8, 1.2) + r.uniform(-20, 20)                    # brightness/contrast
    out = 255 * (np.clip(out, 0, 255) / 255) ** r.uniform(.8, 1.25)        # gamma
    out = np.clip(out, 0, 255).astype(np.uint8); out[~mask] = 0          # keep background black
    if r.random() < .3: out = cv2.GaussianBlur(out, (0, 0), r.uniform(.5, 1.5))          # mild blur (P5)
    if r.random() < .3:
        _, enc = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, int(r.integers(40, 90))]); out = cv2.imdecode(enc, 1)
    if r.random() < .5: out = out[:, ::-1]
    M = cv2.getRotationMatrix2D((256, 256), r.uniform(-180, 180), r.uniform(.95, 1.05))
    return cv2.warpAffine(np.ascontiguousarray(out), M, (512, 512))
base = preprocess(cv2.imread(f"{ROOT}/{df.image[picks[2]]}"))
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
axes[0].imshow(cv2.cvtColor(base, cv2.COLOR_BGR2RGB)); axes[0].set_title("input", fontsize=9)
for k in range(1, 6):
    axes[k].imshow(cv2.cvtColor(augment(base, rng), cv2.COLOR_BGR2RGB)); axes[k].set_title(f"augmented {k}", fontsize=9)
for a in axes: a.axis("off")
save(fig, "09_augmentation_preview")

## 9 · Summary of findings → decisions

In [ ]:
g = df.groupby("dup_group")
summary = {
    "n_images": len(df), "n_train": int((df.split == "train").sum()), "n_test": int((df.split == "test").sum()),
    "class_counts": df.grade.value_counts().sort_index().to_dict(),
    "imbalance_ratio": round(df.grade.value_counts().max() / df.grade.value_counts().min(), 2),
    "label_rule_violations": audit["n"].to_dict(),
    "mild_with_hard_exudate": int(((df.grade == 1) & (df.HE == 1)).sum()),
    "dup_vessel_similarity_threshold": TAU,
    "images_in_dup_groups": int(g.grade.transform("size").gt(1).sum()),
    "dup_groups_cross_split": int((g.split.nunique() > 1).sum()),
    "dup_groups_grade_conflict": int((g.grade.nunique() > 1).sum()),
    "max_abs_spearman_property_vs_grade": assoc["spearman_rho_vs_grade"].abs().max(),
}
print(json.dumps(summary, indent=2, default=int))
json.dump(summary, open(f"{OUT_DIR}/eda_summary.json", "w"), indent=2, default=int)
keep = ["image", "split", "grade", "lr"] + LES + ["n_lesions", "label_flag", "md5", "dup_group"] + PROPS + ["dark_frac"]
df[keep].to_csv(f"{OUT_DIR}/cfp_eda.csv", index=False)
print("saved:", f"{OUT_DIR}/cfp_eda.csv")